#  Baseline Tree Model — DT / RF / XGBoost / LightGBM

## 실험 목표
트리 기반 모델들을 비교, 단계적으로 성능 개선

## 실험 흐름
1. **4모델 비교** (DT/RF/XGB/LGBM) → LGBM 최우수 (RMSE 2.24)
2. **로그 변환** → RMSE 1.99 (-11%)
3. **파생변수** (Intensity, Effort) → RMSE 1.78 (-11%)
4. **Optuna 하이퍼파라미터 튜닝** → RMSE ~1.37 (-23%)
5. **Seed Ensemble (5 seeds)** → 안정성 향상
6. **트리 모델 한계 확인** → Polynomial Regression으로 전환 결정

## 성능 요약
| 단계 | 모델 | RMSE |
|------|------|------|
| Baseline | LGBM | 2.241 |
| + Log transform | LGBM | 1.993 |
| + 파생변수 | LGBM | 1.775 |
| + Optuna | LGBM | ~1.37 |

---
## 실험 1: 라이브러리 임포트 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import random, os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error

# 재현성을 위한 시드 고정
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

# RMSE 계산 함수
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
# 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"\n타겟 통계:\n{train['Calories_Burned'].describe()}")

---
## 실험 2: 4모델 비교 (DT / RF / XGBoost / LightGBM)

트리 기반 모델 4종을 동일 조건에서 비교하여 최적 모델 선정
- 동일한 train/val split (test_size=0.2, random_state=42)
- RMSE 기준으로 평가

In [ ]:
#범주형 변수 인코딩 One-Hot Encoding (X에만!)
# 겟더미로 가져와서 변수 생성(순서 없는 명목형 범주는 One-Hot이 안전)
# 카테고리를 독립적인 변수로 만들어서 순서x,크기왜곡x,모델이 오해x
# LabelEncoder 사용시 숫자 크기에따라 학습시 의미가 생겨버릴수있음 특히 Weight_Status때문에..
train_x = pd.get_dummies(train_x, columns=['Gender', 'Weight_Status'], drop_first=True)
test_x = pd.get_dummies(test_x, columns=['Gender', 'Weight_Status'], drop_first=True)

# train / test 컬럼 맞추기
train_x, test_x = train_x.align(test_x, join='left', axis=1, fill_value=0)
test_x = test_x[train_x.columns]  # train 컬럼 순서대로 맞춤



In [ ]:
# 전처리: 범주형 인코딩
from sklearn.preprocessing import LabelEncoder

# train/test 복사
train_df = train.copy()
test_df = test.copy()

# LabelEncoder 적용 (초기 실험용 — 추후 OneHot으로 변경)
le_gender = LabelEncoder()
le_weight = LabelEncoder()

train_df['Gender'] = le_gender.fit_transform(train_df['Gender'])
train_df['Weight_Status'] = le_weight.fit_transform(train_df['Weight_Status'])
test_df['Gender'] = le_gender.transform(test_df['Gender'])
test_df['Weight_Status'] = le_weight.transform(test_df['Weight_Status'])

# 피처/타겟 분리
feature_cols = [c for c in train_df.columns if c not in ['ID', 'Calories_Burned']]
X = train_df[feature_cols]
y = train_df['Calories_Burned']

# Train/Val 분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")

In [ ]:
# 4종 모델 비교
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

models = {
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
}

# 모델별 학습 및 평가
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    score = rmse(y_val, pred)
    results[name] = score
    print(f"{name:15s} | RMSE: {score:.4f}")

# 최우수 모델 확인
best_model = min(results, key=results.get)
print(f"\n 최우수 모델: {best_model} (RMSE {results[best_model]:.4f})")

### 결과
- **LightGBM**이 가장 낮은 RMSE로 최우수 모델 선정
- DecisionTree < RandomForest < XGBoost < LightGBM 순서로 성능 향상
- 이후 실험은 LGBM 기반으로 진행

---
## 실험 3: 로그 변환 적용

EDA에서 확인된 타겟 우측 편향(skew ~0.82)을 보정하기 위해 log1p 변환 적용
- `log1p(y)` → 학습 → `expm1(pred)` 로 복원

In [ ]:
# 로그 변환 전후 비교
y_train_log = np.log1p(y_train)  # log(1+y): 0값 안전 처리
y_val_log = np.log1p(y_val)

# LGBM + 로그 변환 학습
lgbm_log = LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
lgbm_log.fit(X_train, y_train_log)

# 예측 후 복원
pred_log = lgbm_log.predict(X_val)
pred_original = np.expm1(pred_log)  # exp(pred) - 1: 원래 스케일 복원

# RMSE 비교
rmse_before = results['LightGBM']
rmse_after = rmse(y_val, pred_original)
print(f"로그 변환 전 RMSE: {rmse_before:.4f}")
print(f"로그 변환 후 RMSE: {rmse_after:.4f}")
print(f"개선율: {(rmse_before - rmse_after) / rmse_before * 100:.1f}%")

---
## 실험 4: 파생변수 생성 (Intensity, Effort)

EDA에서 Duration×BPM 상호작용이 가장 높은 상관관계를 보였음
- **Intensity** = Exercise_Duration × BPM (운동 강도)
- **Effort** = Weight × Intensity (에너지 소비 추정량)
- Height 통합: Height(Feet) × 12 + Remainder_Inches

In [ ]:
# 파생변수 생성 함수
def create_features(df):
    df = df.copy()
    # Height 통합 (인치)
    df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    # 운동 강도 = 시간 × 심박수
    df['Intensity'] = df['Exercise_Duration'] * df['BPM']
    # 에너지 소비 = 체중 × 강도
    df['Effort'] = df['Weight(lb)'] * df['Intensity']
    return df

# 파생변수 적용
X_train_feat = create_features(X_train)
X_val_feat = create_features(X_val)

# LGBM + 파생변수 + 로그 변환
lgbm_feat = LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
lgbm_feat.fit(X_train_feat, y_train_log)

pred_feat = np.expm1(lgbm_feat.predict(X_val_feat))
rmse_feat = rmse(y_val, pred_feat)

print(f"파생변수 추가 후 RMSE: {rmse_feat:.4f}")
print(f"개선율: {(rmse_after - rmse_feat) / rmse_after * 100:.1f}%")

---
## 실험 5: Optuna 하이퍼파라미터 튜닝

LightGBM의 주요 파라미터를 Optuna로 자동 탐색
- 탐색 파라미터: n_estimators, max_depth, learning_rate, num_leaves, reg_alpha, reg_lambda 등
- 100회 trial, RMSE 기준 최적화

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 파생변수 적용된 전체 데이터 준비
train_full = create_features(train_df.copy())
X_full = train_full[[c for c in train_full.columns if c not in ['ID', 'Calories_Burned']]]
y_full_log = np.log1p(train_df['Calories_Burned'])

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1
    }
    
    # 5-Fold CV
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_idx, va_idx in kf.split(X_full):
        model = LGBMRegressor(**params)
        model.fit(X_full.iloc[tr_idx], y_full_log.iloc[tr_idx])
        pred = np.expm1(model.predict(X_full.iloc[va_idx]))
        scores.append(rmse(np.expm1(y_full_log.iloc[va_idx]), pred))
    return np.mean(scores)

# 탐색 실행 (100 trials)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\n최적 RMSE: {study.best_value:.4f}")
print(f"최적 파라미터: {study.best_params}")

---
## 실험 6: Seed Ensemble (5 seeds)

동일 하이퍼파라미터로 random seed만 달리하여 5개 모델의 예측을 평균
- 개별 모델의 분산을 줄여 안정적인 예측
- 트리 모델의 노드 분할이 seed에 따라 달라지는 특성 활용

In [ ]:
# Seed Ensemble: 5개 시드로 학습 후 평균
best_params = study.best_params.copy()
seeds = [42, 123, 456, 789, 2024]

# 전체 train으로 학습, test 예측
test_full = create_features(test_df.copy())
X_test = test_full[[c for c in X_full.columns]]

ensemble_preds = []
for seed in seeds:
    best_params['random_state'] = seed
    best_params['verbose'] = -1
    
    model = LGBMRegressor(**best_params)
    model.fit(X_full, y_full_log)
    pred = np.expm1(model.predict(X_test))
    ensemble_preds.append(pred)

# 5개 예측 평균
final_pred = np.mean(ensemble_preds, axis=0)
print(f"Seed Ensemble 완료 (seeds: {seeds})")
print(f"예측 통계: mean={final_pred.mean():.2f}, min={final_pred.min():.2f}, max={final_pred.max():.2f}")

---
## 결론: 트리 모델의 한계

- **최종 성능**: RMSE ~1.37 (Optuna + Seed Ensemble)
- LGBM이 가장 좋았으나, 더 이상의 개선이 어려움
- 데이터의 곱셈 구조(Duration×BPM×Temp)를 트리가 완벽히 학습하기 어려움
- **→ Polynomial Regression으로 모델 전환 결정**

| 시도 | 효과 |
|------|------|
| 4모델 비교 | LGBM 선정 |
| 로그 변환 | ~11% 개선 |
| 파생변수 | ~11% 개선 |
| Optuna | ~23% 개선 |
| Seed Ensemble | 안정성 향상 |
| **한계** | **RMSE ~1.37에서 정체** |